[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C64_ML_Knowledge_QA_Course/02_optimization/02_optimization_qa.ipynb)

# 02 · 优化与训练问答（6 种优化器对拍 / AdamW 解耦衰减 / warmup / 初始化方差 / 损失梯度形状）

目标：把「说说 Adam 和 SGD 的区别」这类问题，从背概念变成**能亲手跑出数字来支撑答案**。

本 notebook 你会亲手实现并验证：
1. **6 种优化器**在同一个「病态曲率」玩具问题上的真实轨迹差异
2. **AdamW 解耦衰减 vs Adam+L2** 的衰减强度差异（量化到具体倍数）
3. **有无 warmup** 时 Adam 早期更新幅度的对比
4. **Xavier vs He 初始化**对深层 ReLU 网络前向方差的影响（20 层追踪）
5. **Focal Loss / Label Smoothing** 的梯度形状与标准 CE 的差异

> 心智模型：**每一个「优化器八卦」都对应一个可以用 20 行 numpy 复现的具体现象——背答案不如背怎么现场证明它。**

## 0 · 环境自检

In [ ]:
import sys
import numpy as np

print('Python:', sys.version.split()[0])
print('numpy :', np.__version__)
assert sys.version_info >= (3, 8)
rng = np.random.default_rng(0)
print('\n✅ 环境自检通过：本课全程 numpy + 标准库，无需 GPU，不联网。')

## 1 · 六种优化器在「病态曲率」玩具问题上的真实差异

玩具问题：$f(x,y) = 0.5(a x^2 + b y^2)$，取 $a{=}1, b{=}100$（条件数 100，是一个典型的窄山谷）。
梯度 $\nabla f = (ax, by)$。从 $(1,1)$ 出发，看谁能把两个方向都收敛好。

In [ ]:
def grad_fn(xy, a=1.0, b=100.0):
    return np.array([a * xy[0], b * xy[1]])

def run_sgd(xy0, lr, steps, a=1.0, b=100.0):
    xy = xy0.copy()
    for _ in range(steps):
        xy = xy - lr * grad_fn(xy, a, b)
    return xy

def run_momentum(xy0, lr, steps, mu=0.9, nesterov=False, a=1.0, b=100.0):
    xy = xy0.copy(); v = np.zeros_like(xy)
    for _ in range(steps):
        g = grad_fn(xy + mu * v, a, b) if nesterov else grad_fn(xy, a, b)
        v = mu * v - lr * g
        xy = xy + v
    return xy

def run_adagrad(xy0, lr, steps, eps=1e-8, a=1.0, b=100.0):
    xy = xy0.copy(); G = np.zeros_like(xy)
    for _ in range(steps):
        g = grad_fn(xy, a, b)
        G += g ** 2
        xy = xy - lr * g / (np.sqrt(G) + eps)
    return xy

def run_rmsprop(xy0, lr, steps, beta=0.9, eps=1e-8, a=1.0, b=100.0):
    xy = xy0.copy(); s = np.zeros_like(xy)
    for _ in range(steps):
        g = grad_fn(xy, a, b)
        s = beta * s + (1 - beta) * g ** 2
        xy = xy - lr * g / (np.sqrt(s) + eps)
    return xy

def run_adam(xy0, lr, steps, beta1=0.9, beta2=0.999, eps=1e-8, a=1.0, b=100.0):
    xy = xy0.copy(); m = np.zeros_like(xy); v = np.zeros_like(xy)
    for t in range(1, steps + 1):
        g = grad_fn(xy, a, b)
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g ** 2
        mhat = m / (1 - beta1 ** t)
        vhat = v / (1 - beta2 ** t)
        xy = xy - lr * mhat / (np.sqrt(vhat) + eps)
    return xy

x0 = np.array([1.0, 1.0])
print('六种优化器实现就位（SGD / Momentum / Nesterov / AdaGrad / RMSProp / Adam）。')

In [ ]:
# ---- 实验 A：病态曲率下，SGD 对两个方向的收敛速度极不平衡，自适应方法几乎完全拉平 ----
steps = 200
xy_sgd = run_sgd(x0, lr=0.008, steps=steps)
xy_adagrad = run_adagrad(x0, lr=0.1, steps=steps)
xy_rmsprop = run_rmsprop(x0, lr=0.1, steps=steps)
xy_adam = run_adam(x0, lr=0.1, steps=steps)

def balance_ratio(xy):
    """y 方向剩余量 / x 方向剩余量：越接近 1 说明两个方向收敛得越"平衡"。"""
    return abs(xy[1]) / abs(xy[0])

r_sgd = balance_ratio(xy_sgd)
r_ada = balance_ratio(xy_adagrad)
r_rms = balance_ratio(xy_rmsprop)
r_adam = balance_ratio(xy_adam)

print(f'SGD      最终位置={xy_sgd}   平衡比 r={r_sgd:.3e}   (b=100a 的高曲率方向几乎瞬间归零，'
      f'低曲率方向严重滞后)')
print(f'AdaGrad  最终位置={xy_adagrad}   平衡比 r={r_ada:.4f}')
print(f'RMSProp  最终位置={xy_rmsprop}   平衡比 r={r_rms:.4f}')
print(f'Adam     最终位置={xy_adam}   平衡比 r={r_adam:.4f}')

assert r_sgd < 1e-6, 'SGD 应表现出极端不平衡（两方向收敛速度差几十个数量级）'
assert abs(r_ada - 1.0) < 0.2, 'AdaGrad 应把两个方向的收敛速度拉到接近 1:1'
assert abs(r_rms - 1.0) < 0.2, 'RMSProp 同理'
assert abs(r_adam - 1.0) < 0.2, 'Adam 同理'
print('\n✅ 验证：自适应方法对每个维度独立按曲率归一化，SGD 没有这个机制。')

In [ ]:
# ---- 实验 B：Momentum / Nesterov 相比 SGD 的加速 —— 与随之而来的震荡代价 ----
steps_b = 60
lr_b = 0.008
sgd_traj = [x0.copy()]
xy = x0.copy()
for _ in range(steps_b):
    xy = xy - lr_b * grad_fn(xy)
    sgd_traj.append(xy.copy())
sgd_traj = np.array(sgd_traj)

def trajectory(runner, **kw):
    xy = x0.copy(); out = [xy.copy()]
    return out

def momentum_traj(lr, steps, mu=0.9, nesterov=False):
    xy = x0.copy(); v = np.zeros_like(xy); out = [xy.copy()]
    for _ in range(steps):
        g = grad_fn(xy + mu * v) if nesterov else grad_fn(xy)
        v = mu * v - lr * g
        xy = xy + v
        out.append(xy.copy())
    return np.array(out)

mom_traj = momentum_traj(lr_b, steps_b, nesterov=False)
nes_traj = momentum_traj(lr_b, steps_b, nesterov=True)

def sign_changes(seq):
    s = np.sign(seq); s = s[s != 0]
    return int(np.sum(s[1:] != s[:-1]))

def loss(xy, a=1.0, b=100.0):
    return 0.5 * (a * xy[0] ** 2 + b * xy[1] ** 2)

loss_sgd, loss_mom, loss_nes = loss(sgd_traj[-1]), loss(mom_traj[-1]), loss(nes_traj[-1])
sc_sgd, sc_mom, sc_nes = sign_changes(sgd_traj[:, 1]), sign_changes(mom_traj[:, 1]), sign_changes(nes_traj[:, 1])

print(f'SGD       final loss={loss_sgd:.5f}   y方向变号次数={sc_sgd}')
print(f'Momentum  final loss={loss_mom:.5f}   y方向变号次数={sc_mom}')
print(f'Nesterov  final loss={loss_nes:.5f}   y方向变号次数={sc_nes}')

assert loss_mom < loss_sgd, 'Momentum 应比 SGD 收敛得更好（低曲率方向被加速）'
assert loss_nes < loss_mom, 'Nesterov 的前瞻修正应进一步改善'
assert sc_mom > sc_sgd, 'Momentum 的加速是有代价的：陡峭方向出现了震荡（变号次数增多）'
print('\n✅ 验证：动量法用"陡峭方向出现震荡"换来了"低曲率方向更快收敛"——这正是它的权衡。')

## 2 · AdamW 的解耦衰减 vs Adam 的 L2：量化衰减强度的偏差

构造两个参数：一个历史梯度方差大（$v_{\text{big}}$），一个历史梯度方差小（$v_{\text{small}}$）。
假设当前真实任务梯度为 0（只看"权重衰减"这一项单独的效果），比较两种做法让参数收缩了多少。

In [ ]:
def adam_decay_step(x, v_prior, wd, lr, beta1=0.9, beta2=0.999, eps=1e-8, t=1000, decoupled=False):
    """单步：任务梯度=0，只看权重衰减项的效果；t 取较大值使 bias correction≈1，避免干扰对比。"""
    if decoupled:
        decay_update = lr * wd * x                 # AdamW：直接作用在参数上，与 v 无关
        return x - decay_update
    g_reg = wd * x                                   # Adam+L2：正则项被当成梯度的一部分
    m = (1 - beta1) * g_reg
    v = beta2 * v_prior + (1 - beta2) * g_reg ** 2   # v_prior 远大于 g_reg^2 时几乎不变
    mhat = m / (1 - beta1 ** t)
    vhat = v / (1 - beta2 ** t)
    return x - lr * mhat / (np.sqrt(vhat) + eps)

wd, lr = 0.01, 0.1
x_start = 1.0
v_big, v_small = 1.0, 1e-4     # 一个"历史梯度方差大"的参数、一个"历史梯度方差小"的参数

x1_l2  = adam_decay_step(x_start, v_big,   wd, lr, decoupled=False)
x2_l2  = adam_decay_step(x_start, v_small, wd, lr, decoupled=False)
x1_adw = adam_decay_step(x_start, v_big,   wd, lr, decoupled=True)
x2_adw = adam_decay_step(x_start, v_small, wd, lr, decoupled=True)

shrink1_l2, shrink2_l2 = (x_start - x1_l2) / x_start, (x_start - x2_l2) / x_start
shrink1_adw, shrink2_adw = (x_start - x1_adw) / x_start, (x_start - x2_adw) / x_start

print(f'Adam+L2   ：v_big 收缩比例={shrink1_l2:.3e}   v_small 收缩比例={shrink2_l2:.3e}   '
      f'两者相差 {shrink2_l2/shrink1_l2:.1f} 倍')
print(f'AdamW     ：v_big 收缩比例={shrink1_adw:.3e}   v_small 收缩比例={shrink2_adw:.3e}   '
      f'两者相差 {shrink2_adw/shrink1_adw:.4f} 倍')

assert shrink2_l2 / shrink1_l2 > 50, 'Adam+L2 下，梯度历史方差小的参数应被过度衰减（相差应远大于1）'
assert abs(shrink2_adw / shrink1_adw - 1.0) < 1e-6, 'AdamW 的衰减率必须与 v 无关，两者应完全相等'
assert np.isclose(shrink1_adw, lr * wd) and np.isclose(shrink2_adw, lr * wd), 'AdamW 的收缩比例应恒等于 lr*wd'
print('\n✅ 验证：Adam+L2 的衰减强度被 v 缩放（相差 100 倍量级），AdamW 对所有参数统一衰减率 lr·wd。')

## 3 · Warmup 为什么必要：Adam 首步更新幅度与训练早期梯度范数

In [ ]:
# ---- 现象 1：Adam 在 t=1 时，更新幅度几乎恒为 lr，与真实梯度的量级无关 ----
def adam_first_step(g1, lr=1.0, beta1=0.9, beta2=0.999, eps=1e-8):
    m = (1 - beta1) * g1
    v = (1 - beta2) * g1 ** 2
    mhat = m / (1 - beta1)      # t=1
    vhat = v / (1 - beta2)
    return lr * mhat / (np.sqrt(vhat) + eps)

print(f"{'|g1|':>10} {'update':>10} {'update/lr':>10}")
for g1 in [1e-3, 1e-1, 1.0, 10.0, 1000.0]:
    u = adam_first_step(g1, lr=1.0)
    print(f'{g1:>10} {u:>10.6f} {u/1.0:>10.6f}')
    assert abs(u - 1.0) < 0.02, f'g1={g1} 时首步更新幅度应仍接近 lr(=1.0)'
print('\n✅ 验证：无论真实梯度是 1e-3 还是 1e3，Adam 第一步的更新幅度都约等于学习率本身——'
      '这是一次"盲目的满幅步"，warmup 就是用来压小它的。')

In [ ]:
# ---- 现象 2：在一个简单非线性问题上，有无 warmup 时训练早期梯度范数的差异 ----
def grad_quartic(x):
    return 4 * x ** 3            # f(x)=x^4，远离 0 的区域梯度增长很快（模拟"走远了梯度暴涨"的损失面）

def adam_run(x0, lr_base, steps, warmup=0, beta1=0.9, beta2=0.999, eps=1e-8):
    x = x0; m = 0.0; v = 0.0
    grad_norms = []
    for t in range(1, steps + 1):
        g = grad_quartic(x)
        grad_norms.append(abs(g))
        lr_t = lr_base * min(1.0, t / warmup) if warmup > 0 else lr_base
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g ** 2
        mhat = m / (1 - beta1 ** t)
        vhat = v / (1 - beta2 ** t)
        x = x - lr_t * mhat / (np.sqrt(vhat) + eps)
        if abs(x) > 1e6:
            break
    return x, grad_norms

x_final_no, gn_no = adam_run(0.1, lr_base=0.5, steps=30, warmup=0)
x_final_yes, gn_yes = adam_run(0.1, lr_base=0.5, steps=30, warmup=10)

max_no, max_yes = max(gn_no), max(gn_yes)
print(f'无 warmup: 最终 x={x_final_no:.4f}   全程最大梯度范数={max_no:.4f}')
print(f'有 warmup: 最终 x={x_final_yes:.4f}   全程最大梯度范数={max_yes:.4f}')
print(f'无 warmup / 有 warmup 的最大梯度范数比 = {max_no/max_yes:.2f}')

assert max_no > max_yes * 10, '不加 warmup 时，早期大步长把 x 推离原点导致梯度暴涨，应显著大于有 warmup 的情形'
print('\n✅ 验证：没有 warmup 时，第一次满幅步就把参数推到了梯度陡增的区域，形成"早期不稳定"的正反馈。')

## 4 · Xavier vs He：20 层前向传播的激活方差追踪

In [ ]:
def forward_track(width=256, depth=20, batch=512, gain=1.0, activation='relu', seed=0):
    r = np.random.default_rng(seed)
    x = r.standard_normal((batch, width))
    variances = [float(np.var(x))]
    for _ in range(depth):
        std = np.sqrt(gain / width)          # fan_in = width
        W = r.standard_normal((width, width)) * std
        z = x @ W
        x = np.maximum(z, 0) if activation == 'relu' else z
        variances.append(float(np.var(x)))
    return variances

v_xavier_relu = forward_track(gain=1.0, activation='relu')      # Xavier 配 ReLU：错误搭配
v_he_relu = forward_track(gain=2.0, activation='relu')          # He 配 ReLU：正确搭配
v_xavier_linear = forward_track(gain=1.0, activation='linear')  # Xavier 配线性：正确搭配

ratio_xavier_relu = v_xavier_relu[-1] / v_xavier_relu[0]
ratio_he_relu = v_he_relu[-1] / v_he_relu[0]
ratio_xavier_linear = v_xavier_linear[-1] / v_xavier_linear[0]

print(f'Xavier(gain=1)+ReLU   : 第0层方差={v_xavier_relu[0]:.4f}  第20层方差={v_xavier_relu[-1]:.6f}  比值={ratio_xavier_relu:.3e}')
print(f'He(gain=2)+ReLU       : 第0层方差={v_he_relu[0]:.4f}  第20层方差={v_he_relu[-1]:.4f}  比值={ratio_he_relu:.4f}')
print(f'Xavier(gain=1)+线性   : 第0层方差={v_xavier_linear[0]:.4f}  第20层方差={v_xavier_linear[-1]:.4f}  比值={ratio_xavier_linear:.4f}')

assert ratio_xavier_relu < 1e-3, 'Xavier 配 ReLU 应导致深层激活方差指数级坍缩'
assert 0.2 < ratio_he_relu < 5.0, 'He 配 ReLU 应把方差维持在同一量级'
assert 0.3 < ratio_xavier_linear < 3.0, 'Xavier 配线性激活本就该保持方差'
print('\n✅ 验证：用错初始化(Xavier+ReLU)会让 20 层后的激活值只剩初始的千分之一以下——'
      '这正是"配错激活函数的初始化"这个踩雷点的真实数值证据。')

## 5 · 损失函数的梯度形状：Huber / Focal / Label Smoothing

In [ ]:
# ---- Huber vs MSE vs MAE 的梯度形状 ----
def huber_grad(e, delta=1.0):
    return np.where(np.abs(e) <= delta, e, delta * np.sign(e))

errors = np.array([-5.0, -2.0, -0.5, 0.0, 0.5, 2.0, 5.0])
mse_grad = errors                      # d/de [0.5 e^2] = e
mae_grad = np.sign(errors)             # d/de |e| = sign(e)
hub_grad = huber_grad(errors, delta=1.0)

print('errors        :', errors)
print('MSE   grad(=e):', mse_grad)
print('MAE   grad    :', mae_grad)
print('Huber grad(δ=1):', hub_grad)

assert np.allclose(mse_grad, errors)
assert np.allclose(mae_grad, np.sign(errors))
assert np.allclose(hub_grad, np.array([-1.0, -1.0, -0.5, 0.0, 0.5, 1.0, 1.0]))
print('\n✅ 验证：|e|<=delta 时 Huber 退化为 MSE 的梯度（e 本身），|e|>delta 时退化为 MAE 的有界梯度（±delta）。')

In [ ]:
# ---- CE vs Focal Loss 的梯度衰减（数值微分，二分类，target=1）----
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def ce_loss(z, y=1.0):
    p = np.clip(sigmoid(z), 1e-12, 1 - 1e-12)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

def focal_loss(z, y=1.0, gamma=2.0):
    p = np.clip(sigmoid(z), 1e-12, 1 - 1e-12)
    pt = p if y == 1.0 else (1 - p)
    return -(1 - pt) ** gamma * np.log(pt)

def numgrad(f, z, eps=1e-5):
    return (f(z + eps) - f(z - eps)) / (2 * eps)

print(f"{'z':>4} {'pt':>7} {'CE_grad':>10} {'Focal_grad':>11} {'比值':>8}")
ratios = {}
for z in [-3, 0, 1, 3, 5]:
    p = sigmoid(z)
    g_ce = numgrad(lambda zz: ce_loss(zz, 1.0), z)
    g_fl = numgrad(lambda zz: focal_loss(zz, 1.0, 2.0), z)
    ratios[z] = g_fl / g_ce
    print(f'{z:>4} {p:>7.4f} {g_ce:>10.5f} {g_fl:>11.5f} {ratios[z]:>8.5f}')

assert ratios[5] < 0.01, '预测很自信且答对(z=5)时，Focal 梯度应被压制到 CE 的百分之一以下'
assert ratios[-3] > 0.5, '预测严重出错(z=-3)的难例，Focal 梯度不应被过度压制'
print('\n✅ 验证：Focal 对"自信且答对"的样本梯度衰减极快，对难例梯度基本不打折——这就是"难例加权"的数值证据。')

In [ ]:
# ---- Label Smoothing：给"越来越自信"踩刹车（sigmoid+BCE 对 logit 的梯度 = p - y）----
def bce_grad_logit(z, y):
    return sigmoid(z) - y

eps_ls = 0.1
y_hard = 1.0
y_smooth = 1 - eps_ls / 2     # 二分类下 label smoothing 目标: (1-eps)*1 + eps*0.5

z_probe = [0, 1, 2, 3, 4, 5]
hard_grads = [bce_grad_logit(z, y_hard) for z in z_probe]
smooth_grads = [bce_grad_logit(z, y_smooth) for z in z_probe]

for z, gh, gs in zip(z_probe, hard_grads, smooth_grads):
    print(f'z={z:>2}  p={sigmoid(z):.4f}  hard_grad={gh:+.5f}  smooth_grad={gs:+.5f}')

assert all(g < 0 for g in hard_grads), '硬标签下梯度应恒为负，持续鼓励 logit 增大（永不满足）'
assert smooth_grads[0] < 0 and smooth_grads[-1] > 0, 'label smoothing 下梯度应在 p 超过 y_smooth 后变号（踩刹车）'
z_cross = np.log(y_smooth / (1 - y_smooth))
assert 2.5 < z_cross < 3.5
print(f'\n✅ 验证：label smoothing 让梯度在 z≈{z_cross:.2f}（p≈{y_smooth}）处变号，'
      f'从"继续推高置信度"变成"往回拉"——硬标签下这件事永远不会发生。')

## ✏️ 练习 1：AdamW 与 Adam+L2 的衰减比率函数

实现 `l2_vs_decoupled_ratio(v_big, v_small, wd, lr, beta1=0.9, t=1000)`，
返回 `(ratio_l2, ratio_decoupled)`：
- `ratio_l2` = Adam+L2 下 `v_small` 参数的收缩比例 / `v_big` 参数的收缩比例（应远大于 1）
- `ratio_decoupled` = AdamW 下同样的比值（应恒等于 1.0，因为与 v 无关）

复用第 2 节已经定义的 `adam_decay_step`。

In [ ]:
def l2_vs_decoupled_ratio(v_big, v_small, wd, lr, beta1=0.9, t=1000):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
r_l2, r_dec = l2_vs_decoupled_ratio(1.0, 1e-4, 0.01, 0.1)
print(f'ratio_l2={r_l2:.2f}   ratio_decoupled={r_dec:.6f}')
assert r_l2 > 50, r_l2
assert abs(r_dec - 1.0) < 1e-6, r_dec
r_l2_b, r_dec_b = l2_vs_decoupled_ratio(4.0, 1e-2, 0.02, 0.2)
assert r_l2_b > 1.0
assert abs(r_dec_b - 1.0) < 1e-6
print('✅ 练习 1 通过：AdamW 的衰减率与参数的历史梯度方差无关，Adam+L2 不是。')

## ✏️ 练习 2：Adam 首步更新幅度的普适性

实现 `adam_first_step_magnitude(g1, lr=1.0, beta1=0.9, beta2=0.999, eps=1e-8)`，
返回 Adam 在 $t=1$ 时对某个梯度值 `g1` 的更新幅度（标量，取绝对值）。

要点：$t=1$ 时的 bias correction 恰好抵消掉 $(1-\beta_1)$ 与 $(1-\beta_2)$，
使得更新幅度约等于 $\eta\cdot\mathrm{sign}(g_1)$ 的量级。

In [ ]:
def adam_first_step_magnitude(g1, lr=1.0, beta1=0.9, beta2=0.999, eps=1e-8):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
for g1 in [1e-4, 1e-2, 1.0, 100.0, 1e4]:
    u = adam_first_step_magnitude(g1, lr=0.3)
    assert abs(u - 0.3) < 0.3 * 0.05, (g1, u)
    print(f'g1={g1:<10} -> update={u:.6f}  (目标 lr=0.3)')
print('✅ 练习 2 通过：跨 8 个数量级的梯度，首步更新幅度都锁定在 lr 附近。')

## ✏️ 练习 3：He / Xavier 的方差公式反推

实现 `required_gain(activation)`，返回让方差在一次线性层后保持不变所需的 `gain`
（其中 $\mathrm{Var}(W)=\mathrm{gain}/\text{fan\_in}$）：
- `'relu'` → 需要补偿 ReLU 砍掉一半激活值 → `gain = 2.0`
- `'linear'` / `'tanh'` → 不需要补偿 → `gain = 1.0`

再实现 `layer_std(fan_in, activation)` = $\sqrt{\mathrm{gain}/\text{fan\_in}}$。

In [ ]:
def required_gain(activation):
    # TODO
    raise NotImplementedError

def layer_std(fan_in, activation):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert required_gain('relu') == 2.0
assert required_gain('linear') == 1.0
assert required_gain('tanh') == 1.0
assert np.isclose(layer_std(256, 'relu'), np.sqrt(2 / 256))
assert np.isclose(layer_std(256, 'linear'), np.sqrt(1 / 256))
assert layer_std(256, 'relu') > layer_std(256, 'linear'), 'He 的标准差应比 Xavier 大（因为要补偿 ReLU 打对折）'
print('layer_std(256, relu)  =', layer_std(256, 'relu'))
print('layer_std(256, linear)=', layer_std(256, 'linear'))
print('✅ 练习 3 通过：He 比 Xavier 多一个恰好 sqrt(2) 倍的标准差，用来补偿 ReLU 砍掉的那一半方差。')

## ✏️ 练习 4：Label Smoothing 的梯度变号临界点

实现 `label_smoothing_crossover(eps, num_classes=2)`，返回二分类 sigmoid+BCE 梯度
由负变正的临界 logit $z^\*=\log\big(\dfrac{y_s}{1-y_s}\big)$，其中
$y_s = (1-\epsilon) + \epsilon/\text{num\_classes}$。

In [ ]:
def label_smoothing_crossover(eps, num_classes=2):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
z_star = label_smoothing_crossover(0.1, num_classes=2)
assert 2.5 < z_star < 3.5, z_star
p_star = sigmoid(z_star)
assert abs(p_star - 0.95) < 1e-6, p_star     # y_smooth = 1-0.1/2 = 0.95
# eps 越大，天花板越低，交叉点应越靠前（越小的 z 就会触发刹车）
z_star_big_eps = label_smoothing_crossover(0.4, num_classes=2)
assert z_star_big_eps < z_star
print(f'eps=0.1 -> z*={z_star:.4f} (p*={p_star:.4f})')
print(f'eps=0.4 -> z*={z_star_big_eps:.4f}')
print('✅ 练习 4 通过：eps 越大，"置信度天花板"越低，模型更早被拉回。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def l2_vs_decoupled_ratio(v_big, v_small, wd, lr, beta1=0.9, t=1000):
    x_start = 1.0
    x1_l2 = adam_decay_step(x_start, v_big, wd, lr, beta1=beta1, t=t, decoupled=False)
    x2_l2 = adam_decay_step(x_start, v_small, wd, lr, beta1=beta1, t=t, decoupled=False)
    x1_dec = adam_decay_step(x_start, v_big, wd, lr, beta1=beta1, t=t, decoupled=True)
    x2_dec = adam_decay_step(x_start, v_small, wd, lr, beta1=beta1, t=t, decoupled=True)
    shrink1_l2 = (x_start - x1_l2) / x_start
    shrink2_l2 = (x_start - x2_l2) / x_start
    shrink1_dec = (x_start - x1_dec) / x_start
    shrink2_dec = (x_start - x2_dec) / x_start
    return shrink2_l2 / shrink1_l2, shrink2_dec / shrink1_dec

In [ ]:
# 练习 2 参考答案
def adam_first_step_magnitude(g1, lr=1.0, beta1=0.9, beta2=0.999, eps=1e-8):
    m = (1 - beta1) * g1
    v = (1 - beta2) * g1 ** 2
    mhat = m / (1 - beta1)
    vhat = v / (1 - beta2)
    return abs(lr * mhat / (np.sqrt(vhat) + eps))

In [ ]:
# 练习 3 参考答案
def required_gain(activation):
    return 2.0 if activation == 'relu' else 1.0

def layer_std(fan_in, activation):
    return np.sqrt(required_gain(activation) / fan_in)

In [ ]:
# 练习 4 参考答案
def label_smoothing_crossover(eps, num_classes=2):
    y_s = (1 - eps) + eps / num_classes
    return np.log(y_s / (1 - y_s))

---
## 🧪 真实工程胶囊：优化器/初始化/损失函数的选型速查 + 常见配置坑

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# A. 优化器选型速查（面试可直接背的一句话版本）
# ══════════════════════════════════════════════════════════════════════
# □ CV 分类/检测 backbone 训练          -> SGD + Momentum(0.9) + cosine/step schedule + warmup
# □ Transformer（视觉或语言）           -> AdamW(beta1=0.9, beta2=0.999 或 0.95) + cosine + warmup
# □ 稀疏特征/推荐系统                   -> AdaGrad 或其变体（天然适合稀疏梯度）
# □ RNN/序列模型的历史遗留代码           -> RMSProp（AdaGrad 的"学习率死亡"问题在这里被绕开）
# □ 不确定用什么 -> 默认起手 AdamW，收敛快、对学习率不敏感、坑最少

# ══════════════════════════════════════════════════════════════════════
# B. Adam/AdamW 常见配置坑
# ══════════════════════════════════════════════════════════════════════
# 坑1: torch.optim.Adam(weight_decay=...) 做的是 L2（不是解耦衰减）
#      -> 要解耦衰减必须显式用 torch.optim.AdamW
# 坑2: BN/LayerNorm 的 weight 和 bias 不应该被 weight decay
#      -> 常见做法：把 1D 参数（norm 层的 gamma/beta、所有 bias）单独分组，weight_decay=0
# 坑3: 混合精度下，优化器状态 (m, v) 与 BN running stats 建议保留 fp32
#      -> 只有前向激活与梯度计算用 fp16/bf16

# ══════════════════════════════════════════════════════════════════════
# C. 初始化速查
# ══════════════════════════════════════════════════════════════════════
# ReLU/LeakyReLU/GELU(近似)      -> He/Kaiming: std = sqrt(2/fan_in)
# tanh/sigmoid/线性输出层         -> Xavier/Glorot: std = sqrt(2/(fan_in+fan_out))
# 残差分支的最后一层                -> 常见技巧：额外把该层权重初始化为 0 或很小
#                                    （让残差块初始时近似恒等映射，训练更稳，见 C64-03）

# ══════════════════════════════════════════════════════════════════════
# D. 混合精度速查
# ══════════════════════════════════════════════════════════════════════
# fp16 + 动态 loss scaling          -> 老一代 GPU（无原生 bf16 支持）的标准选择
# bf16（无需 loss scaling）         -> 新一代 GPU 的默认选择，工程更简单
# 排查"训练出现 NaN"的第一步        -> 检查是不是 loss scale 溢出，看 loss scale 是否被自动减半

# ══════════════════════════════════════════════════════════════════════
# E. 损失函数速查
# ══════════════════════════════════════════════════════════════════════
# 回归 + 数据干净                   -> MSE
# 回归 + 有离群点/标注噪声           -> Huber（检测框回归的 smooth-L1 是它的近亲，见 C54-02）
# 分类 + 类别平衡                   -> 标准 CE
# 分类 + 极度不平衡（检测里的背景/前景）-> Focal Loss
# 分类 + 担心过度自信/需要更好泛化    -> Label Smoothing（但会牺牲部分置信度可解释性，见 C64-04）

# ══════════════════════════════════════════════════════════════════════
# F. 与本课程其他部分的分工（别重复准备）
# ══════════════════════════════════════════════════════════════════════
# · 优化器的完整数学推导（收敛性证明、动量的物理类比）  -> C07 模块 05（本课不重复）
# · Focal Loss / GIoU 等检测专用损失的完整推导与实现     -> C54 模块 02
# · 归一化家族、残差连接、注意力机制                     -> C64 模块 03（下一站）
# · 校准、ROC/PR、A/B 测试                              -> C64 模块 04
'''
print(RECIPE)
for token in ['AdamW', 'He/Kaiming', 'loss scale', 'Focal Loss', 'C07 模块 05', 'C54 模块 02']:
    assert token in RECIPE, token
print('✅ 检查单覆盖：优化器选型 / 配置坑 / 初始化 / 混合精度 / 损失函数 / 课程分工')

### 小结

- **优化器不是背名字，是背「解决了前一个的什么问题、什么时候反而更差」**：Momentum 缓解震荡但可能超调；
  AdaGrad 自适应但学习率会"死掉"；RMSProp 修复了这一点；Adam 加上动量与偏差修正成为默认起手式；
  **AdamW 修复了 Adam 里 L2 正则被自适应分母不均匀缩放的 bug**——这是本模块面试价值最高的单点。
- **Warmup 不是玄学**：Adam 在训练早期（尤其 $t=1$）的更新幅度几乎恒为学习率本身，与真实梯度量级无关——
  这是一次"盲目的满幅步"，warmup 用来给它踩刹车，直到二阶矩估计变得可信。
- **He 比 Xavier 多的那个系数 2，是专门为 ReLU 砍掉一半激活值这件事补的**——配错组合
  （Xavier+ReLU）会让 20 层后的激活方差坍缩到初始值的千分之一以下。
- **批大小不是越大越好**：大 batch 降低了梯度噪声这个"隐式正则化"，需要配合线性缩放学习率 + warmup
  才能追平小 batch 的泛化表现。
- **损失函数选择是一道约束满足题**：离群值多选 Huber/MAE 不选 MSE；类别不平衡选 Focal 不选纯 CE；
  担心过度自信就上 Label Smoothing，但要知道它会牺牲部分置信度的可解释性。

下一站：**模块 03 · 深度学习架构问答** —— 卷积口算、归一化家族、残差连接、
注意力与 Transformer，同样是"60 秒讲清楚 + 接住追问"的打法。